## Проверка гипотезы о читателях из Москвы и Санкт-Петербурга

- Автор: Луцкова Екатерина
- Дата: 17.06.2026

## Цели и задачи проекта

<b>Цель проекта:</b> проверить гипотезу "Пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы". 

<b>Задачи проекта:</b>

* загрузить данные и проверить их на корректность;
* оценить размер групп и их распределение;
* провести статистический тест для проверки гипотезы;
* интепретировать результаты теста.


## Описание данных

Таблица `yandex_knigi_data.csv` содержит данные о пользователях из Москвы и Санкт-Петербурга и их суммарном количестве часов прослушивания/чтения книг. 

Таблица имеет следующие поля:

- `city` — город или регион географического положения;
- `puid` — идентификатор пользователя;
- `hours` — суммарная длительность чтения или прослушивания в часах.


## Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка гипотезы в Python
3. Аналитическая записка

## 1. Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
import pandas as pd
from scipy import stats as st
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest

In [2]:
df_knigi = pd.read_csv('/datasets/yandex_knigi_data.csv')

Проверим, что данные загрузились и выглядят корректно.

In [3]:
df_knigi.head(10)

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434
5,5,Москва,352567,8.206369
6,6,Москва,439493,0.857758
7,7,Москва,494541,0.035072
8,8,Москва,647235,12.000076
9,9,Москва,656480,0.973032


Видим, что в таблице присутствует столбец `Unnamed: 0`, который полностью дублирует индексы. Удалим его.

In [4]:
df_knigi.drop('Unnamed: 0', axis=1)

,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434
...,...,...,...
8779,Санкт-Петербург,1130000028554332,4.107774
8780,Санкт-Петербург,1130000030307246,45.069222
8781,Санкт-Петербург,1130000038726322,0.211944
8782,Санкт-Петербург,1130000047892100,4.311841


Проверим наличие дубликатов в идентификаторах пользователей.

In [5]:
df_knigi['puid'].duplicated().sum()

244

Выявлено 244 дубликата в id пользователей. 

Предположим, что дубликаты возникли из-за того, что один и тот же пользователь заходил в приложение из обоих городов, а значит система зафиксировала активность и в Москве, и в Спб. 

Далее попробуем проверить это предположение.

Рассчитаем количество уникальных пользователей в каждой из наблюдаемых групп.

In [6]:
msk_group = df_knigi[df_knigi['city']=='Москва']['puid'].nunique() 
spb_group = msk_group = df_knigi[df_knigi['city']=='Санкт-Петербург']['puid'].nunique() 
print(msk_group, spb_group)

2550 2550


Видим, что количество пользователей в группах равно, а значит распределение равномерное.

Узнаем, нет ли пересечений в группах, и убедимся, что никто из пользователей случайно не попал в обе группы одновременно.

In [7]:
msk_group = df_knigi[df_knigi['city']=='Москва']['puid']
spb_group = df_knigi[df_knigi['city']=='Санкт-Петербург']['puid']
intersection = list(set(msk_group) & set(spb_group))
print(len(intersection))

244


Видим, что 244 пользователя попали в обе группы. Таким образом, наша догадка подтверждается - дубликаты возникли из-за нахождения одних и тех же пользователей и в Москве, и в Спб. Можем смело удалить данные дубликаты.

In [8]:
df_knigi = df_knigi.drop_duplicates()

## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
# Данные по Москве
msk_group_1 = df_knigi[df_knigi['city']=='Москва']['hours']
# Данные по Спб
spb_group_2 = df_knigi[df_knigi['city']=='Санкт-Петербург']['hours']

# Уровень статистической значимости
alpha = 0.05 

results = st.ttest_ind(
    msk_group_1, 
    spb_group_2,
    alternative='less' 
)

print('p-значение:', results.pvalue)

if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу')
else:
    print('Подтверждаем нулевую гипотезу')

p-значение: 0.21101894136116772
Подтверждаем нулевую гипотезу


## 3. Аналитическая записка

Для сравнения двух выборочных средних был выбран статистический t-тест Стьюдента (с двумя выборками). Выбираем данный тест, поскольку наша гипотеза касается средних показателей.

Уровень статистической значимости - 0.05.

По результатам теста p-value равно 0.21101894136116772. 

Интерпретируем результат: p-value больше уровня значимости, из чего следует, что нулевая гипотеза подтверждается. Выборочные средние двух групп статистически равны, а значит, средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается. 

Возможная причина таких результатов заключается в том, что разница между средними показателями прослушиваний не столь значима для подтверждения альтернативной гипотезы и может объясняться случайными колебаниями.
